# create a clm nc and fill it with ucla-roms output

In [9]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
from cmocean import cm
import os
import netCDF4 as nc
import glob
from tqdm import tqdm


In [14]:
import netCDF4
from datetime import datetime
import os

def make_clm_from_cdl(his_file, clm_file):

    his = netCDF4.Dataset(his_file)
    clm = netCDF4.Dataset(clm_file, 'w', format='NETCDF4')

    # ---------- Global attributes ----------
    clm.Description = "CLimatology for offline simulation"
    clm.Author = "Siming"
    clm.Created = "2026-04-16T10:00:00"
    clm.type = "ROMS CLM file"

    # ---------- Dimensions ----------
    xi_rho = 602
    xi_u   = 601
    xi_v   = 602
    xi_psi = 601
    
    eta_rho = 402
    eta_u   = 402
    eta_v   = 401
    eta_psi = 401

    s_rho = 45
    s_w   = 46

    ntime = 744
    
    dims_size = {
    'xi_rho': xi_rho,
    'xi_u': xi_u,
    'xi_v': xi_v,
    'xi_psi': xi_psi,
    'eta_rho': eta_rho,
    'eta_u': eta_u,
    'eta_v': eta_v,
    'eta_psi': eta_psi,
    's_rho': s_rho,
    's_w': s_w,
    'ocean_time': ntime
}

    clm.createDimension('xi_rho', xi_rho)
    clm.createDimension('xi_u', xi_u)
    clm.createDimension('xi_v', xi_v)
    clm.createDimension('xi_psi', xi_psi)

    clm.createDimension('eta_rho', eta_rho)
    clm.createDimension('eta_u', eta_u)
    clm.createDimension('eta_v', eta_v)
    clm.createDimension('eta_psi', eta_psi)

    clm.createDimension('N', s_rho)
    clm.createDimension('s_rho', s_rho)
    clm.createDimension('s_w', s_w)

    clm.createDimension('tracer', 2)
    clm.createDimension('boundary', 4)

    clm.createDimension('ocean_time', ntime)
    clm.createDimension('zeta_time', ntime)
    clm.createDimension('u2d_time', ntime)
    clm.createDimension('v2d_time', ntime)
    clm.createDimension('u3d_time', ntime)
    clm.createDimension('v3d_time', ntime)
    clm.createDimension('temp_time', ntime)
    clm.createDimension('salt_time', ntime)
    clm.createDimension('srf_time', ntime)
    clm.createDimension('ssf_time', ntime)

    # ---------- Time variables ----------
    def add_time_var(name):
        v = clm.createVariable(name, 'f8', (name,))
        v.units = "seconds since 2000-01-01 00:00:00"
        v.calendar = "gregorian_proleptic"
        return v

    zeta_time = add_time_var('zeta_time')
    zeta_time.long_name = "time for free-surface"
    zeta_time.field = "zeta_time, scalar, series"

    u2d_time = add_time_var('u2d_time')
    u2d_time.long_name = "time for vertically integrated u-momentum component"
    u2d_time.field = "u2d_time, scalar, series"

    v2d_time = add_time_var('v2d_time')
    v2d_time.long_name = "time for vertically integrated v-momentum component"
    v2d_time.field = "v2d_time, scalar, series"

    u3d_time = add_time_var('u3d_time')
    u3d_time.long_name = "time for u-momentum component"
    u3d_time.field = "u3d_time, scalar, series"

    v3d_time = add_time_var('v3d_time')
    v3d_time.long_name = "time for v-momentum component"
    v3d_time.field = "v3d_time, scalar, series"

    ocean_time = add_time_var('ocean_time')
    ocean_time.long_name = "time for S-coordinate vertical momentum component"
    ocean_time.field = "ocean_time, scalar, series"

    temp_time = add_time_var('temp_time')
    temp_time.long_name = "time for potential temperature"
    temp_time.field = "temp_time, scalar, series"

    salt_time = add_time_var('salt_time')
    salt_time.long_name = "time for salinity"
    salt_time.field = "salt_time, scalar, series"

    srf_time = add_time_var('srf_time')
    srf_time.long_name = "time for solar shortwave radiation flux"
    srf_time.field = "srf_time, scalar, series"

    ssf_time = add_time_var('ssf_time')
    ssf_time.long_name = "time for surface net heat flux"
    ssf_time.field = "ssf_time, scalar, series"

    # ---------- Write time values ----------
    ocean_time[:] = his.variables['ocean_time'][:]
    zeta_time[:] = his.variables['ocean_time'][:]
    u2d_time[:] = his.variables['ocean_time'][:]
    v2d_time[:] = his.variables['ocean_time'][:]
    u3d_time[:] = his.variables['ocean_time'][:]
    v3d_time[:] = his.variables['ocean_time'][:]
    temp_time[:] = his.variables['ocean_time'][:]
    salt_time[:] = his.variables['ocean_time'][:]
    srf_time[:] = his.variables['ocean_time'][:]
    ssf_time[:] = his.variables['ocean_time'][:]

    # ---------- Physical variables (streaming write) ----------
    def add_var(name, dims):
        # 把字符串维度名转换成对应的维度大小
        chunk_sizes = [1]  # 时间维
        for d in dims[1:]:
            chunk_sizes.append(dims_size[d])

        return clm.createVariable(
            name, 'f8', dims,
            chunksizes=tuple(chunk_sizes),
            zlib=True,
            complevel=1
        )

    zeta = add_var('zeta', ('ocean_time', 'eta_rho', 'xi_rho'))
    zeta.long_name = "free-surface"
    zeta.units = "meter"
    zeta.field = "free-surface, scalar, series"
    zeta.time = "zeta_time"

    ubar = add_var('ubar', ('ocean_time', 'eta_u', 'xi_u'))
    ubar.long_name = "vertically integrated u-momentum component"
    ubar.units = "meter second-1"
    ubar.field = "ubar-velocity, scalar, series"
    ubar.time = "u2d_time"

    vbar = add_var('vbar', ('ocean_time', 'eta_v', 'xi_v'))
    vbar.long_name = "vertically integrated v-momentum component"
    vbar.units = "meter second-1"
    vbar.field = "vbar-velocity, scalar, series"
    vbar.time = "v2d_time"

    u = add_var('u', ('ocean_time', 's_rho', 'eta_u', 'xi_u'))
    u.long_name = "u-momentum component"
    u.units = "meter second-1"
    u.field = "u-velocity, scalar, series"
    u.time = "u3d_time"

    v = add_var('v', ('ocean_time', 's_rho', 'eta_v', 'xi_v'))
    v.long_name = "v-momentum component"
    v.units = "meter second-1"
    v.field = "v-velocity, scalar, series"
    v.time = "v3d_time"

    omega = add_var('omega', ('ocean_time', 's_w', 'eta_rho', 'xi_rho'))
    omega.long_name = "S-coordinate vertical momentum component"
    omega.units = "meter second-1"
    omega.field = "omega, scalar, series"
    omega.time = "ocean_time"

    temp = add_var('temp', ('ocean_time', 's_rho', 'eta_rho', 'xi_rho'))
    temp.long_name = "potential temperature"
    temp.units = "Celsius"
    temp.field = "temperature, scalar, series"
    temp.time = "temp_time"

    salt = add_var('salt', ('ocean_time', 's_rho', 'eta_rho', 'xi_rho'))
    salt.long_name = "salinity"
    salt.field = "salinity, scalar, series"
    salt.time = "salt_time"

    swrad = add_var('swrad', ('ocean_time', 'eta_rho', 'xi_rho'))
    swrad.long_name = "solar shortwave radiation flux"
    swrad.units = "watt meter-2"
    swrad.field = "shortwave radiation, scalar, series"
    swrad.time = "srf_time"

    shflux = add_var('shflux', ('ocean_time', 'eta_rho', 'xi_rho'))
    shflux.long_name = "surface net heat flux"
    shflux.units = "watt meter-2"
    shflux.field = "surface heat flux, scalar, series"
    shflux.time = "ssf_time"

    # ---------- Streaming write ----------
    for t in tqdm(range(ntime)):
        if t % 50 == 0:
            print(f'Writing time step {t}/{ntime}')

        zeta[t] = his.variables['zeta'][t]
        ubar[t] = his.variables['ubar'][t]
        vbar[t] = his.variables['vbar'][t]
        u[t] = his.variables['u'][t]
        v[t] = his.variables['v'][t]
        omega[t] = his.variables['omega'][t]
        temp[t] = his.variables['temp'][t]
        salt[t] = his.variables['salt'][t]

        # ⚠️ 如果你的输入文件没有 swrad / shflux，用 0 填充
        if 'swrad' in his.variables:
            swrad[t] = his.variables['swrad'][t]
        else:
            swrad[t] = 0.0

        if 'shflux' in his.variables:
            shflux[t] = his.variables['shflux'][t]
        else:
            shflux[t] = 0.0

    clm.close()
    his.close()
    print('✅ CLM file created successfully')

if __name__ == '__main__':
    make_clm_from_cdl(
        his_file='/leader/user/zsm/TWS2_L1_ucla_new/TWS_L1_his_201903_full.nc',
        clm_file='/leader/user/zsm/twc_clm.nc'
    )

  0%|                                                                                                                  | 0/744 [00:00<?, ?it/s]

Writing time step 0/744


  7%|██████▉                                                                                                | 50/744 [07:34<1:49:03,  9.43s/it]

Writing time step 50/744


 13%|█████████████▋                                                                                        | 100/744 [15:10<1:36:49,  9.02s/it]

...

Writing time step 650/744


 94%|███████████████████████████████████████████████████████████████████████████████████████████████▉      | 700/744 [1:40:28<05:47,  7.90s/it]

Writing time step 700/744


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████| 744/744 [1:46:47<00:00,  8.61s/it]


✅ CLM file created successfully


In [15]:
era_path='/pacific1/user/biyesheng_zsm/TWS_L1_input/TWS_era2019010203.nc'
era=xr.open_dataset(era_path)

/tmp/ipykernel_481252/3183158651.py:2: FutureWarning: In a future version of xarray decode_timedelta will default to False rather than None. To silence this warning, set decode_timedelta to True, False, or a 'CFTimedeltaCoder' instance.
  era=xr.open_dataset(era_path)


In [18]:
era

<xarray.Dataset> Size: 14GB
Dimensions:    (time: 2137, eta_rho: 402, xi_rho: 602, rad_time: 2137)
Coordinates:
  * time       (time) timedelta64[ns] 17kB 6940 days 00:00:00 ... 7029 days 0...
  * rad_time   (rad_time) timedelta64[ns] 17kB 6940 days 00:00:00 ... 7029 da...
Dimensions without coordinates: eta_rho, xi_rho
Data variables:
    rain       (time, eta_rho, xi_rho) float32 2GB ...
    uwnd       (time, eta_rho, xi_rho) float32 2GB ...
    vwnd       (time, eta_rho, xi_rho) float32 2GB ...
    wnd_time   (time) timedelta64[ns] 17kB ...
    Tair       (time, eta_rho, xi_rho) float32 2GB ...
    qair       (time, eta_rho, xi_rho) float32 2GB ...
    tair_time  (time) timedelta64[ns] 17kB ...
    qair_time  (time) timedelta64[ns] 17kB ...
    swrad      (rad_time, eta_rho, xi_rho) float32 2GB ...
    lwrad      (rad_time, eta_rho, xi_rho) float32 2GB ...
Attributes:
    title:                 ROMS surface forcing file created by ROMS-Tools
    roms_tools_version:    3.0.1.dev7+gbba77f9
    start_time:            2019-01-01 00:00:00
    end_time:              2019-03-31 00:00:00
    source:                ERA5
    correct_radiation:     True
    wind_dropoff:          True
    use_coarse_grid:       False
    model_reference_date:  2000-01-01 00:00:00
    type:                  physics

In [21]:
swrad=era['swrad'].values[1392:-1,:,:]

In [22]:
swrad.shape

(744, 402, 602)

In [24]:
flx_path='/leader/user/zsm/TWS2_L1_ucla_new/TWS_L1_flx_full.nc'
flx=xr.open_dataset(flx_path)
flx

<xarray.Dataset> Size: 3GB
Dimensions:     (time: 744, eta_rho: 402, xi_u: 601, eta_v: 401, xi_rho: 602)
Coordinates:
  * time        (time) float64 6kB 0.0 0.0 0.0 0.0 0.0 ... 0.0 0.0 0.0 0.0 0.0
Dimensions without coordinates: eta_rho, xi_u, eta_v, xi_rho
Data variables:
    ocean_time  (time) float64 6kB ...
    sustr       (time, eta_rho, xi_u) float32 719MB ...
    svstr       (time, eta_v, xi_rho) float32 718MB ...
    shflx       (time, eta_rho, xi_rho) float32 720MB ...
    ssflx       (time, eta_rho, xi_rho) float32 720MB ...
Attributes: (12/48)
    CDI:                Climate Data Interface version 2.4.4 (https://mpimet....
    Conventions:        CF-1.6
    global_x:           600
    global_y:           400
    title:              Test for Iceland.
    grid_file:          /pacific1/user/biyesheng_zsm/TWS_L1_input/TWS2_L1_gri...
    ...                 ...
    pipe_frc_options:   OFF
    particle_options:   OFF
    git_version:        bfa4c0a5b663cd7b44f06647f2f3e6313d76c7ac
    type:               surface flux                              history
    history:            Thu Apr 16 21:40:34 2026: cdo mergetime TWS_L1_flx_hi...
    CDO:                Climate Data Operators version 2.4.4 (https://mpimet....

In [25]:
sustr=flx['sustr'].values
svstr=flx['svstr'].values
shflux=flx['shflx'].values*3985*1027.5

In [28]:
# clm=nc.Dataset('/leader/user/zsm/twc_clm.nc')

In [34]:
# clm.close()

In [35]:
clm = netCDF4.Dataset(
    '/leader/user/zsm/twc_clm.nc',
    'a'   # 'a' = append / write
)

shflux_var = clm.variables['shflux']
swrad_var = clm.variables['swrad']


ntime = shflux.shape[0]

for t in range(ntime):
    if t % 50 == 0:
        print(f"Writing shflux time {t}/{ntime}")

    shflux_var[t, :, :] = shflux[t, :, :].astype(np.float64)
    swrad_var[t, :, :] = swrad[t, :, :].astype(np.float64)


clm.close()

print("shflux written and CLM file saved successfully")

Writing shflux time 0/744
Writing shflux time 50/744
Writing shflux time 100/744
Writing shflux time 150/744
Writing shflux time 200/744
Writing shflux time 250/744
Writing shflux time 300/744
Writing shflux time 350/744
Writing shflux time 400/744
Writing shflux time 450/744
Writing shflux time 500/744
Writing shflux time 550/744
Writing shflux time 600/744
Writing shflux time 650/744
Writing shflux time 700/744
shflux written and CLM file saved successfully


# create forcing file

In [45]:
import netCDF4
import numpy as np
from datetime import datetime

def make_frc_shell(frc_file, his_file):

    # ---------- 打开历史文件以获取时间和维度 ----------
    his = netCDF4.Dataset(his_file)
    ntime = his.dimensions['time'].size

    clm = netCDF4.Dataset(frc_file, 'w', format='NETCDF4')

    # ---------- Global attributes ----------
    clm.Description = "Forcing for offline simulation"
    clm.Created = datetime.now().strftime("2026-04-16T%H:%M:%S")
    clm.type = "ROMS FRC file"

    # ---------- Dimensions ----------
    xi_rho = 602
    eta_rho = 402
    xi_u = 601
    eta_u = 402
    xi_v = 602
    eta_v = 401

    clm.createDimension('srf_time', ntime)
    clm.createDimension('eta_rho', eta_rho)
    clm.createDimension('xi_rho', xi_rho)

    clm.createDimension('shf_time', ntime)
    clm.createDimension('bhf_time', ntime)

    clm.createDimension('sms_time', ntime)
    clm.createDimension('eta_u', eta_u)
    clm.createDimension('xi_u', xi_u)
    clm.createDimension('eta_v', eta_v)
    clm.createDimension('xi_v', xi_v)

    # ---------- Time variables ----------
    def add_time_var(name):
        v = clm.createVariable(name, 'f8', (name,))
        # ✅ 按照 ocean_frc.nc 标准设置时间单位
        v.units = "seconds since 2000-01-01 00:00:00"
        v.calendar = "gregorian_proleptic"
        return v

    srf_time = add_time_var('srf_time')
    srf_time.long_name = "time for solar shortwave radiation flux"
    srf_time.field = "srf_time, scalar, series"

    shf_time = add_time_var('shf_time')
    shf_time.long_name = "time for surface net heat flux"
    shf_time.field = "shf_time, scalar, series"

    bhf_time = add_time_var('bhf_time')
    bhf_time.long_name = "time for bottom net heat flux"
    bhf_time.field = "bhf_time, scalar, series"

    sms_time = add_time_var('sms_time')
    sms_time.long_name = "time for surface u-momentum stress"
    sms_time.field = "sms_time, scalar, series"

    # ---------- Write Time Values ----------
    # 从 history 文件拷贝 ocean_time
    srf_time[:] = his.variables['ocean_time'][:]
    shf_time[:] = his.variables['ocean_time'][:]
    bhf_time[:] = his.variables['ocean_time'][:]
    sms_time[:] = his.variables['ocean_time'][:]

    # ---------- Forcing variables ----------
    def add_frc_var(name, dims):
        return clm.createVariable(name, 'f8', dims, zlib=True, complevel=1)

    # Solar Radiation
    swrad = add_frc_var('swrad', ('srf_time', 'eta_rho', 'xi_rho'))
    swrad.long_name = "solar shortwave radiation flux"
    swrad.units = "watt meter-2"
    swrad.field = "shortwave radiation, scalar, series"
    swrad.time = "srf_time"

    # Surface Heat Flux
    shflux = add_frc_var('shflux', ('shf_time', 'eta_rho', 'xi_rho'))
    shflux.long_name = "surface net heat flux"
    shflux.units = "watt meter-2"
    shflux.field = "surface heat flux, scalar, series"
    shflux.time = "shf_time"

    # Bottom Heat Flux
    bhflux = add_frc_var('bhflux', ('bhf_time', 'eta_rho', 'xi_rho'))
    bhflux.long_name = "bottom net heat flux"
    bhflux.units = "watt meter-2"
    bhflux.field = "bottom heat flux, scalar, series"
    bhflux.time = "bhf_time"

    # Surface U-Stress
    sustr = add_frc_var('sustr', ('sms_time', 'eta_u', 'xi_u'))
    sustr.long_name = "surface u-momentum stress"
    sustr.units = "newton meter-2"
    sustr.field = "surface u-momentum stress, scalar, series"
    sustr.time = "sms_time"

    # Surface V-Stress
    svstr = add_frc_var('svstr', ('sms_time', 'eta_v', 'xi_v'))
    svstr.long_name = "surface v-momentum stress"
    svstr.units = "newton meter-2"
    svstr.field = "surface v-momentum stress, scalar, series"
    svstr.time = "sms_time"

    # ---------- Fill all forcing variables with 0 ----------
    print(f"Filling forcing variables with zeros (shape: {ntime}, {eta_rho}, {xi_rho})...")

    # 使用切片赋值，因为数据量小（全是0），不会触发 HDF error
    # 对于大文件，如果想更安全，可以像之前那样用循环
    swrad[:, :, :] = 0.0
    shflux[:, :, :] = 0.0
    bhflux[:, :, :] = 0.0
    sustr[:, :, :] = 0.0
    svstr[:, :, :] = 0.0

    # ---------- Close files ----------
    clm.close()
    his.close()
    print("✅ FRC file created with zeros successfully")

if __name__ == '__main__':
    make_frc_shell(
        frc_file='/leader/user/zsm/tws_frc_offline.nc',
        his_file='/leader/user/zsm/TWS2_L1_ucla_new/TWS_L1_his_201903_full.nc'
    )

Filling forcing variables with zeros (shape: 744, 402, 602)...
✅ FRC file created with zeros successfully


In [46]:
frc = netCDF4.Dataset(
    '/leader/user/zsm/tws_frc_offline.nc',
    'a'   # 'a' = append / write
)

shflux_var1 = frc.variables['shflux']
swrad_var1 = frc.variables['swrad']
sustr_var1 = frc.variables['sustr']
svstr_var1 = frc.variables['svstr']

ntime = shflux.shape[0]

shflux_var1[:, :, :] = shflux.astype(np.float64)
swrad_var1[:, :, :] = swrad.astype(np.float64)
sustr_var1[:, :, :] = sustr.astype(np.float64)
svstr_var1[:, :, :] = svstr.astype(np.float64)

# for t in tqdm(range(ntime)):
#     if t % 50 == 0:
#         print(f"Writing shflux time {t}/{ntime}")

#     shflux_var1[t, :, :] = shflux[t, :, :].astype(np.float64)
#     swrad_var1[t, :, :] = swrad[t, :, :].astype(np.float64)
#     sustr_var1[t, :, :] = sustr[t, :, :].astype(np.float64)
#     svstr_var1[t, :, :] = svstr[t, :, :].astype(np.float64)
    



frc.close()

print("shflux, swrad,sustr,svstr written and FRC file saved successfully")

shflux, swrad,sustr,svstr written and FRC file saved successfully


In [43]:
sustr.shape,svstr.shape

((744, 402, 601), (744, 401, 602))